# pycensuskr 분석 갤러리

In [ ]:
from pycensuskr import CensusKR
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

census = CensusKR()


## Example 1: Spatial autocorrelation of economic activity

This section uses `libpysal` and `esda`. Install once if needed.


In [ ]:
# Uncomment once if needed:
# !pip install libpysal esda

from libpysal.weights import Queen
from esda.moran import Moran, Moran_Local

adm2_2020 = census.load_districts(2020)
df_econ = census.anycensus(year=2020, type='economy')
sf = adm2_2020.merge(df_econ[['adm2_code', 'company_total_cnt']], on='adm2_code', how='inner')

w = Queen.from_dataframe(sf)
w.transform = 'r'

y = sf['company_total_cnt'].fillna(0).to_numpy()

m = Moran(y, w)
print('Global Moran I:', m.I)
print('p-value:', m.p_sim)

lm = Moran_Local(y, w)
mean_y = y.mean()

def lisa_class(val, z, p):
    if p > 0.05:
        return 'Not significant'
    if val > mean_y and z > 0:
        return 'High-High'
    if val < mean_y and z > 0:
        return 'Low-Low'
    if val > mean_y and z < 0:
        return 'High-Low'
    return 'Low-High'

sf['cluster'] = [lisa_class(v, z, p) for v, z, p in zip(y, lm.Is, lm.p_sim)]

ax = sf.plot(column='cluster', categorical=True, legend=True, figsize=(8, 6), linewidth=0.1, edgecolor='gray')
ax.set_title('LISA cluster map of company units (2020)')
ax.set_axis_off()
plt.show()


## Example 2: Population change by sex and district (small multiples)


In [ ]:
long_2020 = census.load_data(2020)
pop = long_2020.query("type == 'population' and class1 == 'all households' and class2 in ['male','female']").copy()

pop['value'] = pd.to_numeric(pop['value'], errors='coerce') / 1000

# Simple district trend preview (top-20 districts by total population)
top = (
    pop.query("class2 == 'male'")
    .groupby('adm2_code', as_index=False)['value'].sum()
    .nlargest(20, 'value')['adm2_code']
)
plot_df = pop[pop['adm2_code'].isin(top)]

fig, ax = plt.subplots(figsize=(9, 5))
for (adm2, sex), grp in plot_df.groupby(['adm2', 'class2']):
    ax.plot(grp['year'], grp['value'], alpha=0.5)
ax.set_title('Population trends (sample districts, by sex)')
ax.set_xlabel('Year')
ax.set_ylabel('Population (thousands)')
plt.show()
